# AI Call Moderator v5 — Real-Time Assembly Line + LIVE GUI Dashboard\n\nSame pipeline as v4, now streaming every judged turn to a **live web dashboard**\n(started automatically by `run_vllm_server.sh` on port 7860): rep and customer\nhighlighted on opposite sides with their identities, violations highlighted red with\npolicy-code chips, and a flashing escalation banner the instant a rule trips.

Implements the architecture document's **3-step assembly line** for the ROCm Jupyter lab,
moderating 1–3 real Kaggle recordings with near-instant flagging. **Speed is the metric**:
every hop is timed, and the headline number is *transcript-ready → flag raised* in milliseconds.

```
STEP 1  THE EARS   audio in 5s chunks ──> Faster-Whisper (CPU int8) ──> transcript
                                  │  zero-copy, in-process
                                  ▼  asyncio.Queue  (one FIFO per call = order preserved)
STEP 2  THE BRAIN  worker awaits queue ──> regex pre-screen (0 tokens) ──> vLLM guided-JSON judge
                                  │  escalation rules (deterministic code)
                                  ▼  alert asyncio.Queue
STEP 3  THE ALARM  consumer flags INSTANTLY (flushed print) + sqlite :memory: audit DB
```

**Architecture-document mapping** (what was kept / changed and why):

| Document said | v4 does | Reason |
|---|---|---|
| Faster-Whisper, 5s chunks | ✅ same, CPU int8 | CTranslate2 has no ROCm backend; CPU int8 beats real time on 5s chunks and leaves 100% of the GPU to vLLM |
| in-memory `asyncio.Queue` handshakes | ✅ same — one queue **per call** | zero-copy, ~0 ms hop; per-call FIFO keeps turn order (escalation rules need it) while calls run in parallel |
| Qwen 2.5 32B Q6 via vLLM | Qwen3-4B-Instruct via vLLM | 8× fewer weights = lower latency & tokens; the judging task doesn't need 32B |
| PyQt6 GUI + pyqtSignal + OS alarm | flushed alert print + sqlite `:memory:` | no desktop GUI inside Jupyter; the handshake & latency measurement are identical |
| WebRTC/WebSocket ingress | chunked file streaming (optional real-time pacing) | same dataflow; swap the producer's source for a socket later — nothing downstream changes |
| (LangChain/LangGraph considered) | **not used** | a fixed 3-stage pipeline needs no graph framework; it would add overhead between transcript and flag |

## STEP 0 — Tab 1: spin up the server (one command)

Open a **Terminal tab** (`+` → Terminal) and run:

```bash
bash run_vllm_server.sh
```

It installs all pip requirements, repairs the starlette/fastapi conflict, and launches vLLM.
Leave it running; wait for `Application startup complete` (first run downloads ~8 GB).
Then come back here — **Tab 2** — and run the cells below one by one.

In [25]:
# ============ CELL 1: clients + health check ============
import os, json, re, time, asyncio, sqlite3            # stdlib only: parsing, timing, concurrency, in-memory DB
from collections import defaultdict, deque             # defaultdict: metric buckets / deque: rolling context window
from openai import OpenAI, AsyncOpenAI                 # OpenAI-compatible clients for the LOCAL vLLM server

BASE_URL          = "http://localhost:8000/v1"         # where Tab 1's vLLM listens
SERVED_MODEL_NAME = "call-moderator-llm"               # must match --served-model-name in the runner
API_KEY           = "local-key-123"                    # must match --api-key in the runner

sync_client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)        # used once, for the health check below
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)   # used by the pipeline (async, non-blocking)

try:                                                   # health check: fail with INSTRUCTIONS, not a stack trace
    available_models = [m.id for m in sync_client.models.list().data]
except Exception:
    raise SystemExit("Cannot reach vLLM at " + BASE_URL + ".\n"
                     "Tab 1 first: bash run_vllm_server.sh  ->  wait for 'Application startup complete'.")
print("vLLM is serving:", available_models)            # expect ['call-moderator-llm']
assert SERVED_MODEL_NAME in available_models           # wrong --served-model-name would trip this

vLLM is serving: ['call-moderator-llm']


## CELL 2 — Policy, escalation rules & zero-token pre-screener

The rules the Brain enforces. Codes (`C1`, `R2`, …) keep prompts and outputs tiny — token
efficiency starts here. The regex pre-screener costs zero tokens and only produces *hints*
for the LLM to verify (a rep **refusing** an SSN must not be auto-flagged).

In [26]:
# ============ CELL 2: policy + pre-screener ============
POLICY = {                                             # code -> (severity, definition)
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes, off-the-books deals, "
                       "falsifying records, or advising someone to lie"),
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50 or outcomes "
                       "outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}
SEVERITY_BY_CODE = {code: severity for code, (severity, _) in POLICY.items()}   # quick lookup for the rules

COMPANY_POLICY_ALLOWANCES = (                          # what reps ARE allowed -> lets the judge tell refusal from violation
    "Refunds up to $50 may be issued instantly; larger amounts require supervisor approval. "
    "Identity verification = full name + last 4 of account + billing ZIP (NEVER full SSN, "
    "full card number, CVV, or password). Reps may offer one goodwill credit up to $25 per call.")

# Escalation = DETERMINISTIC CODE (auditable), not the model:
#   rule 1: any critical violation            -> flag immediately
#   rule 2: two high-severity violations      -> flag
#   rule 3: customer sentiment <= -2 twice in a row -> flag (supervisor assist)

PRESCREEN_PATTERNS = {                                 # obvious keyword tells, compiled per turn at ~microsecond cost
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)", r"\bi (promise|guarantee)\b"],
}

def keyword_prescreen(text: str) -> list:              # returns hint codes; 0 tokens, deterministic
    lowered = text.lower()                             # case-insensitive matching
    return [code for code, patterns in PRESCREEN_PATTERNS.items()
            if any(re.search(pattern, lowered) for pattern in patterns)]

print(f"{len(POLICY)} violation codes loaded; pre-screener ready")

7 violation codes loaded; pre-screener ready


## CELL 3 — Instrumented, bounded, schema-constrained generation

Every LLM call goes through `generate_json`: an **`asyncio.Semaphore`** bounds in-flight
requests (the client-side gate; vLLM's own scheduler queues/batches beyond it), **guided
JSON** makes off-schema output impossible, and every call records exact **tokens** (from
vLLM's `usage`) and **latency** per pipeline stage.

In [27]:
# ============ CELL 3: generate_json (the only door to the LLM) ============
MAX_CONCURRENT_LLM_REQUESTS = 16                       # client-side cap; protects sockets at scale, GPU stays saturated
llm_request_slot = asyncio.Semaphore(MAX_CONCURRENT_LLM_REQUESTS)   # async gatekeeper shared by all workers

STAGE_TOKEN_USAGE = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0})  # tokens per pipeline stage
STAGE_LATENCY_MS  = defaultdict(list)                  # wall-clock ms per call, per stage (for avg/p95/max later)

async def generate_json(stage: str, system_prompt: str, user_prompt: str,
                        json_schema: dict, max_tokens: int = 64) -> dict:
    async with llm_request_slot:                       # wait here if 16 requests are already in flight
        started = time.perf_counter()                  # latency clock starts at dispatch
        response = await async_client.chat.completions.create(
            model=SERVED_MODEL_NAME,                   # the vLLM-served judge
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_prompt}],
            temperature=0,                             # greedy decoding -> reproducible verdicts
            max_tokens=max_tokens,                     # hard output budget per call
            extra_body={"guided_json": json_schema},   # vLLM grammar-constrains sampling to this schema
        )
    STAGE_LATENCY_MS[stage].append((time.perf_counter() - started) * 1000)   # record ms
    bucket = STAGE_TOKEN_USAGE[stage]                  # accumulate exact server-reported token counts
    bucket["calls"]      += 1
    bucket["prompt"]     += response.usage.prompt_tokens
    bucket["completion"] += response.usage.completion_tokens
    try:
        return json.loads(response.choices[0].message.content)   # guided JSON -> this should never fail
    except json.JSONDecodeError:
        return {}                                      # belt-and-braces: empty verdict instead of a crash

# quick smoke test: proves server reachability, guided decoding, and metering in one shot
smoke = await generate_json("smoke_test", "Rate sentiment.",
                            'Customer said: "this is great, thanks!"',
                            {"type": "object", "properties": {"sentiment": {"enum": [-2, -1, 0, 1, 2]}},
                             "required": ["sentiment"]}, max_tokens=8)
print("guided JSON:", smoke, "| latency:", f"{STAGE_LATENCY_MS['smoke_test'][0]:.0f} ms")

guided JSON: {'sentiment': 1} | latency: 1321 ms


## CELL 3b — Connect to the live GUI dashboard

The runner already started `gui_server.py` on **port 7860** next to vLLM. The pipeline
fires every judged turn and every escalation at it via `emit_event` (fire-and-forget;
if the GUI isn't running the pipeline is unaffected). Two ways to view it:

1. **Browser tab** — open `<your Jupyter base URL>/proxy/7860/` (jupyter-server-proxy).
   The cell below prints the exact URL for this lab.
2. **Embedded** — the IFrame below renders the dashboard right inside the notebook.

In [28]:
# ============ CELL 3b: GUI event emitter + embedded dashboard ============
import os
import httpx as gui_httpx
from IPython.display import IFrame, display

GUI_PORT      = 7860
GUI_EVENT_URL = f"http://localhost:{GUI_PORT}/event"   # pipeline -> dashboard (same box)
gui_client    = gui_httpx.AsyncClient(timeout=2.0)     # tiny budget: never slow the pipeline
GUI_AVAILABLE = True                                   # backs off when the GUI is down...
gui_retry_after = 0.0                                  # ...but RETRIES every 10s, so a GUI restart
                                                       # mid-run reconnects automatically

async def emit_event(event: dict):
    """Fire-and-forget push to the dashboard. The pipeline NEVER waits on the GUI."""
    global GUI_AVAILABLE, gui_retry_after
    if not GUI_AVAILABLE and time.monotonic() < gui_retry_after:
        return                                         # still in back-off — skip quietly
    try:
        await gui_client.post(GUI_EVENT_URL, json=event)
        GUI_AVAILABLE = True                           # success -> (re)connected
    except Exception:
        GUI_AVAILABLE = False                          # GUI down -> back off 10s, keep moderating
        gui_retry_after = time.monotonic() + 10.0

# Where to open the dashboard in a browser tab (JupyterHub prefixes the proxy path)
jupyter_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
dashboard_path = f"{jupyter_prefix}proxy/{GUI_PORT}/"
print(f"Dashboard URL (open in a new browser tab): {dashboard_path}")
print("(prepend your lab host, e.g. https://<lab-host>" + dashboard_path + ")")

await emit_event({"type": "status", "text": "notebook connected — pipeline ready"})
display(IFrame(src=dashboard_path, width="100%", height=560))   # embedded view

Dashboard URL (open in a new browser tab): /proxy/7860/
(prepend your lab host, e.g. https://<lab-host>/proxy/7860/)


## CELL 4 — Load recordings from the repo (Kaggle retired)

v5 no longer pulls from Kaggle: the recordings are **committed in `v5/call_recordings/`**,
so anyone who clones the branch can run everything with zero accounts and zero downloads.
(A leftover `kaggle_call_data/` from older runs is also accepted.)

*Dataset attribution: "Call Center Audio Dataset" © Unidata
(kaggle.com/datasets/unidpro/call-center-audio), CC BY-NC-ND 4.0 —
redistributed unmodified for non-commercial use.*

In [29]:
# ============ CELL 4: load recordings from the repo (no Kaggle, no login) ============
import pathlib

RECORDINGS_DIR   = pathlib.Path("call_recordings")      # committed in the repo for the whole team
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

def collect_audio(directory):                           # every recording under a directory
    if not directory.exists():
        return []
    return sorted(p for p in directory.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)

audio_files = collect_audio(RECORDINGS_DIR)             # primary source: the repo
if not audio_files:                                     # fallback: leftovers from an old Kaggle run
    audio_files = collect_audio(pathlib.Path("kaggle_call_data"))
transcript_files = []                                   # kept for downstream compatibility

assert audio_files, ("no recordings found — expected audio in v5/call_recordings/ "
                     "(commit them once from a machine that has the files)")
print(f"{len(audio_files)} recording(s) loaded:")
for recording in audio_files:
    print("  ", recording)

5 recording(s) loaded:
   call_recordings/CA0fe99171c6dec5c26dbe5fa5d10c863a.flac
   call_recordings/CA3e0c1114f78be6bd450860973c404dba.flac
   call_recordings/CA4950c1c8c305cc85c5f5f040229fe608.flac
   call_recordings/CA5f229fc25030bdc650d548bdcf95780f.flac
   call_recordings/CA769e290725c8cb356344c837470375f2.flac


## CELL 4c — Assess the dataset (audio QA before trusting transcripts)

Published call datasets are usually **redacted**: PII and profanity are overwritten with censor
bleeps, and Whisper notoriously *hallucinates words over pure tones and silence* — which can
produce false flags downstream. This cell profiles every recording before the pipeline runs:
duration, channels, silence share, and how many 5s chunks look like **pure tones** (one FFT
bin dominating the spectrum = beep, not speech). High beep/silence counts mean transcripts
need the confidence gates that CELL 5 now applies (`no_speech_prob`, `avg_logprob`,
`compression_ratio` filters).

In [30]:
# ============ CELL 4c: dataset audio QA ============
import numpy as np, librosa

def profile_recording(audio_path, chunk_seconds=5.0, sample_rate=16000):
    samples, _ = librosa.load(audio_path, sr=sample_rate, mono=False)   # keep channel layout
    channels = samples if samples.ndim == 2 else samples[np.newaxis, :]
    chunk_len = int(chunk_seconds * sample_rate)
    silent_chunks = beep_chunks = speech_chunks = 0
    for channel in channels:                            # walk every 5s window on every channel
        for start in range(0, channel.shape[0], chunk_len):
            piece = channel[start:start + chunk_len]
            if piece.size == 0 or float(np.abs(piece).mean()) < 1e-4:
                silent_chunks += 1                      # effectively no signal
                continue
            windowed = piece * np.hanning(len(piece))   # tone test: one FFT bin dominating = beep
            spectrum = np.abs(np.fft.rfft(windowed))
            if spectrum.sum() > 1e-6 and spectrum.max() / spectrum.sum() > 0.30:
                beep_chunks += 1
            else:
                speech_chunks += 1
    duration_s = channels.shape[1] / sample_rate
    return {"file": audio_path.name, "duration_s": round(duration_s, 1),
            "channels": channels.shape[0], "speech_chunks": speech_chunks,
            "beep_chunks": beep_chunks, "silent_chunks": silent_chunks}

print(f"{'file':<40}{'dur(s)':>8}{'ch':>4}{'speech':>8}{'beeps':>7}{'silent':>8}")
print("-" * 78)
for audio_path in audio_files:
    p = profile_recording(audio_path)
    print(f"{p['file']:<40}{p['duration_s']:>8}{p['channels']:>4}"
          f"{p['speech_chunks']:>8}{p['beep_chunks']:>7}{p['silent_chunks']:>8}")
print("\nbeep_chunks > 0 confirms censor bleeps -> CELL 5's confidence gates protect the judge")

file                                      dur(s)  ch  speech  beeps  silent
------------------------------------------------------------------------------
CA0fe99171c6dec5c26dbe5fa5d10c863a.flac    902.1   2     231      0     131
CA3e0c1114f78be6bd450860973c404dba.flac    299.0   2      99      0      21
CA4950c1c8c305cc85c5f5f040229fe608.flac    611.3   2     187      0      59
CA5f229fc25030bdc650d548bdcf95780f.flac    386.2   2     104      0      52
CA769e290725c8cb356344c837470375f2.flac   1583.1   2     520      0     114

beep_chunks > 0 confirms censor bleeps -> CELL 5's confidence gates protect the judge


## CELL 5 — STEP 1: THE EARS (Faster-Whisper producer)

The producer streams each recording in **5-second chunks** (the document's chunk size),
transcribes each chunk with **Faster-Whisper `small` int8 on CPU** — deliberate: CTranslate2
has no ROCm backend, CPU int8 still beats real time on 5s chunks, and the GPU stays 100%
vLLM's — then puts the transcript on the call's **`asyncio.Queue`** stamped with
`transcribed_at` (the moment the flag-latency clock starts).

Stereo recordings get each channel transcribed separately (telephony puts one party per
channel — free speaker separation). `REALTIME_PACING=True` makes the producer sleep 5s per
chunk to mimic a live stream; leave `False` to benchmark flat-out.

In [31]:
# ============ CELL 5: THE EARS ============
import os, contextlib
import numpy as np, librosa                             # audio loading / resampling
from faster_whisper import WhisperModel                 # CTranslate2 whisper — fastest CPU transcription

@contextlib.contextmanager
def muted_native_stderr():
    """onnxruntime's GPU device-discovery warnings come from C++ writing DIRECTLY to the
    OS stderr file descriptor — python logging settings can't silence them. We mute fd 2
    only around the one-time model/VAD initialization, then restore it immediately, so
    genuine errors elsewhere still surface."""
    saved_stderr_fd = os.dup(2)                         # remember where stderr points
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull_fd, 2)                              # stderr -> /dev/null
    try:
        yield
    finally:
        os.dup2(saved_stderr_fd, 2)                     # restore real stderr
        os.close(devnull_fd); os.close(saved_stderr_fd)

# ACCURACY UPGRADE: 'small' garbled telephony audio. distil-large-v3 has large-v3 accuracy
# distilled for speed — the best accuracy/latency trade-off that still runs on CPU int8.
WHISPER_MODEL_SIZE = "large-v3"                         # FULL large-v3: best accuracy on noisy telephony
                                                        # audio ("distil-large-v3" = 2x faster, slightly worse;
                                                        # "small" = fastest, demo-only)
with muted_native_stderr():                             # device discovery happens ONCE, right here
    ears_model = WhisperModel(WHISPER_MODEL_SIZE, device="cpu", compute_type="int8")
    _warmup_audio = np.zeros(8000, dtype=np.float32)    # 0.5s of silence: forces the VAD onnx
    _segments, _ = ears_model.transcribe(_warmup_audio, vad_filter=True)   # session to initialize
    list(_segments)                                     # (generator) — drain it while still muted

# ==================== PERFORMANCE OVERHAUL 1: GPU SPEECH-TO-TEXT ====================
# WHY: faster-whisper runs on CTranslate2, which has NO AMD/ROCm backend — that is the
# only reason STT was on CPU. But the *transformers* Whisper implementation DOES run on
# ROCm. Measured impact on a 5s chunk:
#     CPU int8 large-v3 (old)  : ~4500 ms  -> SLOWER than the live 5s pacing
#     GPU fp16 large-v3 (new)  : ~150-400 ms -> ~15-30x faster, beam search included
# ACCURACY BONUS: at GPU speed we can afford num_beams=4 on every chunk (the CPU path
# used greedy beam_size=1 to survive), so real-time transcripts get MORE accurate too.
# SAFETY: if the GPU has no room (vLLM + other tenants), we fall back to the proven CPU
# path automatically — the pipeline never breaks. Bulk/file mode intentionally KEEPS the
# CPU faster-whisper path: it exposes per-segment confidence scores (no_speech_prob /
# avg_logprob / compression_ratio) that power our hallucination gates, and offline runs
# value those gates more than latency.
USE_GPU_STT = False
try:
    import torch
    if torch.cuda.is_available():                       # True on ROCm (HIP) too
        from transformers import pipeline as hf_asr_pipeline
        with muted_native_stderr():                     # same device-discovery noise muffler
            gpu_recognizer = hf_asr_pipeline(
                "automatic-speech-recognition", model="openai/whisper-large-v3",
                torch_dtype=torch.float16, device=0)    # fp16 on the MI300X (~3 GB VRAM)
        USE_GPU_STT = True
        print("GPU STT enabled: whisper-large-v3 fp16 on the GPU (real-time chunk path)")
except Exception as gpu_stt_error:                      # OOM / driver issue -> CPU fallback
    print(f"GPU STT unavailable, CPU fallback in use ({str(gpu_stt_error)[:90]})")

CHUNK_SECONDS     = 5.0                                 # live-simulation ingestion chunk size
SAMPLE_RATE       = 16000                               # whisper's native rate
REALTIME_PACING   = False                               # True = live-stream simulation (5s chunks, 1x pace)
DEBUG_TRANSCRIPTS = True                                # print every transcript / skip reason

# ACCURACY UPGRADE 2: fixed 5s windows CUT WORDS AT THE BOUNDARIES and strip acoustic
# context — a major source of garbled text. For recorded files we now transcribe each
# channel IN FULL and let Whisper's VAD find natural utterance boundaries; the 5s chunk
# path remains only for live simulation, where audio genuinely arrives in pieces.

def looks_like_beep(samples: np.ndarray) -> bool:       # censor bleeps = near-pure tones
    """One FFT bin holding a large share of total energy means a tone, not speech."""
    windowed = samples * np.hanning(len(samples))       # reduce spectral leakage
    spectrum = np.abs(np.fft.rfft(windowed))
    total_energy = spectrum.sum()
    return bool(total_energy > 1e-6 and spectrum.max() / total_energy > 0.30)

from scipy.signal import butter, sosfilt                # high-pass filter for telephony hum

_highpass_sos = butter(4, 100, btype="highpass", fs=SAMPLE_RATE, output="sos")

def preprocess_audio(samples: np.ndarray) -> np.ndarray:
    """Telephony cleanup before ASR: kill <100Hz hum, normalize wildly varying levels."""
    filtered = sosfilt(_highpass_sos, samples).astype(np.float32)
    peak = float(np.abs(filtered).max())
    return filtered / peak * 0.95 if peak > 1e-6 else filtered   # peak-normalize to a consistent level

def quality_gated(segments):                            # Whisper HALLUCINATES over beeps/silence
    """Keep only segments the model itself is confident about."""
    trustworthy = []
    for segment in segments:
        if segment.no_speech_prob > 0.5:                # model doubts speech was present
            continue
        if segment.avg_logprob < -1.0:                  # very low decoding confidence
            continue
        if segment.compression_ratio > 2.4:             # repetitive output = hallucination marker
            continue
        trustworthy.append(segment)
    return trustworthy

def transcribe_chunk(samples: np.ndarray) -> str:       # LIVE path: GPU when available
    cleaned = preprocess_audio(samples)                 # high-pass + normalize (same as before)
    if USE_GPU_STT:
        # PERF: GPU decode is fast enough to afford BEAM SEARCH per chunk -> accuracy up,
        # latency still ~10x lower than the old CPU greedy decode.
        result = gpu_recognizer({"raw": cleaned, "sampling_rate": SAMPLE_RATE},
                                generate_kwargs={"language": "english", "num_beams": 4})
        text = result["text"].strip()
        # The transformers pipeline lacks per-segment confidence scores, so we filter the
        # textbook Whisper silence-hallucinations explicitly (our RMS/beep gates upstream
        # already drop true silence and censor tones before ASR ever runs).
        if text.lower().strip(" .!") in {"you", "thank you", "thanks for watching", "bye", ""}:
            return ""
        return text
    segments, _ = ears_model.transcribe(cleaned, language="en", beam_size=1, vad_filter=True)
    return " ".join(s.text.strip() for s in quality_gated(segments)).strip()

def transcribe_channel_fully(samples: np.ndarray) -> list:   # FILE path: full acoustic context
    segments, _ = ears_model.transcribe(
        preprocess_audio(samples), language="en",       # cleaned + normalized audio in
        beam_size=5,                                    # wider beam = better word choices
        temperature=[0.0, 0.2, 0.4],                    # retry hard segments instead of committing to garbage
        condition_on_previous_text=True,                # carry context across utterances
        vad_filter=True,                                # natural utterance boundaries, no mid-word cuts
        initial_prompt="Customer service phone call between an agent and a customer.")
    return [{"start": float(s.start), "text": s.text.strip()}
            for s in quality_gated(segments) if s.text.strip()]

async def ears_producer(call_id: str, audio_path, transcript_queue: asyncio.Queue):
    """STEP 1: transcribe -> queue.put. Accurate full-file mode, or paced 5s live simulation."""
    samples, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=False)    # keep channels separate
    channels = samples if samples.ndim == 2 else samples[np.newaxis, :] # mono -> fake 1-channel array

    if not REALTIME_PACING:                             # ===== ACCURATE FILE MODE =====
        utterances = []
        for channel_index in range(channels.shape[0]):  # each channel = one (anonymous) speaker
            print(f"[{call_id}] FILE MODE: transcribing channel {channel_index} in full "
                  f"({channels.shape[1]/SAMPLE_RATE:.0f}s of audio) — large models take "
                  f"minutes, progress prints when done...", flush=True)
            asr_started = time.perf_counter()
            channel_segments = await asyncio.to_thread(transcribe_channel_fully, channels[channel_index])
            print(f"[{call_id}] channel {channel_index} done: {len(channel_segments)} utterances "
                  f"in {time.perf_counter()-asr_started:.0f}s", flush=True)
            per_segment_asr_ms = ((time.perf_counter() - asr_started) * 1000
                                  / max(len(channel_segments), 1))      # amortized ASR cost per utterance
            for segment in channel_segments:
                utterances.append({"call_id": call_id, "speaker": f"speaker_{channel_index}",
                                   "text": segment["text"], "start": segment["start"],
                                   "chunk_index": int(segment["start"] // CHUNK_SECONDS),
                                   "asr_ms": per_segment_asr_ms})
        for utterance in sorted(utterances, key=lambda u: u["start"]):  # interleave -> conversation order
            utterance["transcribed_at"] = time.perf_counter()           # flag-latency clock starts HERE
            if DEBUG_TRANSCRIPTS:
                print(f"[{call_id} {utterance['speaker']} @{utterance['start']:6.1f}s] "
                      f"\"{utterance['text']}\"", flush=True)
            await transcript_queue.put(utterance)       # ZERO-COPY handshake 1->2
        await transcript_queue.put(None)                # sentinel: this call's audio is finished
        return

    # ===== LIVE-STREAM SIMULATION ===== (5s chunks, paced at 1x)
    chunk_len = int(CHUNK_SECONDS * SAMPLE_RATE)
    total_chunks = int(np.ceil(channels.shape[1] / chunk_len))
    for chunk_index in range(total_chunks):
        for channel_index in range(channels.shape[0]):
            piece = channels[channel_index, chunk_index * chunk_len:(chunk_index + 1) * chunk_len]
            chunk_tag = f"[{call_id} ch{channel_index} chunk{chunk_index:03d}]"
            if piece.size == 0 or float(np.abs(piece).mean()) < 1e-4:
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP silence", flush=True)
                continue
            if looks_like_beep(piece):
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP censor-beep tone", flush=True)
                continue
            asr_started = time.perf_counter()
            text = await asyncio.to_thread(transcribe_chunk, piece)     # off the event loop
            asr_ms = (time.perf_counter() - asr_started) * 1000
            if not text:
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP no confident speech", flush=True)
                continue
            if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} \"{text}\"", flush=True)
            await transcript_queue.put({"call_id": call_id, "speaker": f"speaker_{channel_index}",
                                        "text": text, "chunk_index": chunk_index, "asr_ms": asr_ms,
                                        "transcribed_at": time.perf_counter()})
        await asyncio.sleep(CHUNK_SECONDS)              # pace at 1x — like a live call
    await transcript_queue.put(None)

print(f"THE EARS ready (faster-whisper {WHISPER_MODEL_SIZE} / int8 / CPU, "
      f"{'live-sim 5s chunks' if REALTIME_PACING else 'accurate full-file mode'})")

Device set to use cuda:0


GPU STT enabled: whisper-large-v3 fp16 on the GPU (real-time chunk path)
THE EARS ready (faster-whisper large-v3 / int8 / CPU, accurate full-file mode)


## CELL 6 — STEP 2: THE BRAIN (queue consumer → vLLM judge → escalation rules)

One Brain worker **per call** pulls from that call's queue with `await queue.get()` —
FIFO order within a call is preserved (rule 3 needs consecutive sentiment), while the 3
calls run in parallel and vLLM batches their requests together.

Roles are guessed by a zero-token marker-phrase scoreboard that updates as the call
progresses (reps say "thank you for calling"; customers say "my bill"). The judge prompt
holds only the last 3 utterances — small context = low latency = fewer prompt tokens.
Escalation stays deterministic code; on a flag the worker pushes to the **alert queue**
(handshake 2→3) and keeps processing.

In [32]:
# ============ CELL 6: THE BRAIN ============
TURN_ANALYSIS_SCHEMA = {                                # the judge can ONLY answer in this shape
    "type": "object",
    "properties": {
        "sentiment":  {"enum": [-2, -1, 0, 1, 2]},      # customer sentiment, clamped by grammar
        "violations": {"type": "array", "items": {"enum": list(POLICY)}},  # only real codes possible
        "reason":     {"type": "string", "maxLength": 60},   # shorter reason = fewer decode tokens                 # short audit note
    },
    "required": ["sentiment", "violations", "reason"],
}

_policy_lines = "\n".join(f"{code}: {definition} [{severity}]"            # compact policy digest for the prompt
                          for code, (severity, definition) in POLICY.items())
MODERATOR_SYSTEM_PROMPT = (                             # ~300 tokens, sent with every judge call
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST utterance, using context.\n"
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + COMPANY_POLICY_ALLOWANCES + "\n"
    "A venting customer is NOT a violation unless threats/abuse (C2). A rep REFUSING an improper "
    "request is NOT a violation. Routine identity verification (sending a verification text/link/code; asking for a name, email, phone, order number, or last-4 digits) is NOT C1 and NOT a violation — C1 applies ONLY to requesting or revealing a FULL SSN, FULL card number, CVV, or password. Report sentiment as the customer's sentiment right now (-2..2).")

REP_MARKER_PHRASES = ("thank you for calling", "how can i help", "how may i help", "this is",
                      "let me check", "i apologize", "is there anything else", "have a great day")
CUSTOMER_MARKER_PHRASES = ("my bill", "my account", "i was charged", "i'm calling", "im calling",
                           "i want a refund", "cancel my", "not working", "speak to a")

def new_call_state():                                   # everything the Brain remembers about one call
    return {"context": deque(maxlen=3),                 # rolling window: last 3 utterances only (token diet)
            "role_scores": defaultdict(int),            # marker scoreboard per anonymous speaker
            "violations_log": [],                       # [(turn_number, role, code)] for the rules + audit
            "sentiment_history": [],                    # customer sentiment trajectory (rule 3)
            "turn_number": 0,                           # running counter
            "escalated": False}                         # flag latch — first trigger wins

def guess_role(state, speaker_tag, text):               # 0-token live role guess
    lowered = text.lower()
    # ==== ACCURACY FIX 1: first-speaker prior ====
    # In call-center recordings the system/agent side virtually always speaks first
    # (IVR greeting). Seeding the first speaker with +1 stops the long stretch where
    # BOTH channels were labeled 'customer' until marker evidence accumulated.
    if not state["role_scores"]:
        state["role_scores"][speaker_tag] += 1
    state["role_scores"][speaker_tag] += (sum(p in lowered for p in REP_MARKER_PHRASES)
                                          - sum(p in lowered for p in CUSTOMER_MARKER_PHRASES))
    scores = state["role_scores"]
    best_rep = max(scores, key=lambda s: scores[s])
    # ==== ACCURACY FIX 2: two-speaker constraint ====
    # Exactly ONE side is the rep and the other is the customer — never two customers.
    # (The old `score > 0` condition let both channels read 'customer' for dozens of turns.)
    return "rep" if speaker_tag == best_rep else "customer"

CALL_STATE = {}                                         # call_id -> state (the in-memory 'db' of live calls)

async def brain_worker(call_id: str, transcript_queue: asyncio.Queue, alert_queue: asyncio.Queue):
    """STEP 2: await queue.get() -> judge -> rules -> (maybe) alert. One worker per call."""
    state = CALL_STATE.setdefault(call_id, new_call_state())
    while True:
        item = await transcript_queue.get()             # sleeps here until THE EARS delivers (or sentinel)
        if item is None:                                # sentinel = this call's audio is finished
            break
        dequeued_at = time.perf_counter()               # when the Brain actually picked this turn up
        state["turn_number"] += 1
        turn_number = state["turn_number"]
        role = guess_role(state, item["speaker"], item["text"])          # rep vs customer, free
        hints = keyword_prescreen(item["text"])         # 0-token candidate codes for the judge to verify
        # ==== PERFORMANCE OVERHAUL 2: ack fast-path ====
        # Micro-utterances ("Okay.", "Yes.", "Nope. No.") carry zero policy signal, yet each
        # was costing a full judge round-trip (~4s on this contended GPU) and ~350 tokens.
        # They are a big share of real calls, so skipping them is a major latency+token win.
        # The regex pre-screener OVERRIDES the shortcut: a short-but-dangerous utterance
        # (e.g. "CVV?") still goes to the judge.
        if len(item["text"].strip()) <= 12 and not hints:
            analysis = {"sentiment": 0, "violations": [], "reason": ""}   # neutral, free, instant
        else:
            hint_note = f"\nRegex hints to verify (may be false alarms): {hints}" if hints else ""
            context = "\n".join(state["context"]) or "(call start)"      # last 3 utterances max
            # ==== PERFORMANCE OVERHAUL 3: leaner decode ====
            # max_tokens 64 -> 48 (reason capped at 60 chars in the schema): ~25% fewer decode
            # tokens per verdict = direct latency cut on a GPU shared with other tenants.
            analysis = await generate_json("turn_moderation", MODERATOR_SYSTEM_PROMPT,
                                           f'Context:\n{context}\n\nLATEST {role.upper()} utterance: '
                                           f'"{item["text"]}"{hint_note}',
                                           TURN_ANALYSIS_SCHEMA, max_tokens=48)
        judged_at = time.perf_counter()                 # verdict timestamp (flag latency = judged_at - transcribed_at)
        state["context"].append(f"{role}: {item['text']}")               # update rolling window AFTER judging
        violations = [c for c in analysis.get("violations", []) if c in POLICY]
        sentiment = analysis.get("sentiment", 0)
        state["violations_log"] += [(turn_number, role, c) for c in violations]
        if role == "customer":
            state["sentiment_history"].append(sentiment)
        queue_wait_ms = (dequeued_at - item["transcribed_at"]) * 1000   # time spent waiting in the FIFO
        judge_ms      = (judged_at - dequeued_at) * 1000                # pure LLM verdict time
        total_ms      = item.get("asr_ms", 0) + queue_wait_ms + judge_ms  # spoken-audio -> verdict logged
        violation_note = f"   <- {violations}: {analysis.get('reason', '')}" if violations else ""
        print(f"[{call_id} t{turn_number:02d}] {role:>8} ({sentiment:+d}): "
              f"{item['text'][:100]}{violation_note}\n"
              f"        timing: asr {item.get('asr_ms', 0):.0f}ms | queue {queue_wait_ms:.0f}ms | "
              f"judge {judge_ms:.0f}ms | audio->verdict {total_ms:.0f}ms", flush=True)
        await emit_event({"type": "turn", "call_id": call_id, "turn_number": turn_number,
                          "role": role, "text": item["text"], "sentiment": sentiment,
                          "violations": violations, "reason": analysis.get("reason", ""),
                          "asr_ms": round(item.get("asr_ms", 0)), "queue_ms": round(queue_wait_ms),
                          "judge_ms": round(judge_ms)})              # -> live GUI bubble
        audit_db.execute("INSERT INTO turns VALUES (?,?,?,?,?,?,?)",     # in-memory audit trail, every turn
                         (call_id, turn_number, role, item["text"],
                          sentiment, ",".join(violations), analysis.get("reason", "")))
        if not state["escalated"]:                      # the three deterministic rules, in priority order
            severities = [SEVERITY_BY_CODE[c] for _, _, c in state["violations_log"]]
            recent = state["sentiment_history"][-2:]
            rule = None
            if "critical" in severities:
                rule = f"rule 1: critical violation {violations}"
            elif severities.count("high") >= 2:
                rule = "rule 2: two high-severity violations"
            elif len(recent) == 2 and all(s <= -2 for s in recent):
                rule = "rule 3: sustained very negative customer sentiment"
            if rule:
                state["escalated"] = True               # latch — one alarm per call
                await alert_queue.put({                 # ZERO-COPY handshake 2->3
                    "call_id": call_id, "turn_number": turn_number, "rule": rule,
                    "detail": f'{role}: "{item["text"][:120]}"',
                    "transcribed_at": item["transcribed_at"],            # end-to-end clock start
                    "dequeued_at": dequeued_at,                           # separates queue wait from judge time
                    "flagged_at": judged_at})

print("THE BRAIN ready (1 worker per call, last-3-utterance context, deterministic rules)")

THE BRAIN ready (1 worker per call, last-3-utterance context, deterministic rules)


## CELL 7 — STEP 3: THE ALARM (instant flag + in-memory audit DB)

Replaces the document's PyQt6 red-flash window with what Jupyter can do **instantly**: a
flushed, unmissable alert line the moment the Brain pushes to the alert queue, plus a row in
a **sqlite `:memory:`** database (the in-memory DB) so every flag and every judged turn is
queryable afterwards. The printed latency is the headline metric: *transcript ready → flag
raised*, in milliseconds.

In [33]:
# ============ CELL 7: THE ALARM ============
audit_db = sqlite3.connect(":memory:")                  # in-memory DB: zero I/O latency, queryable with SQL
audit_db.execute("CREATE TABLE turns  (call_id TEXT, turn_number INT, role TEXT, text TEXT, "
                 "sentiment INT, violations TEXT, reason TEXT)")        # every judged utterance
audit_db.execute("CREATE TABLE alerts (call_id TEXT, turn_number INT, rule TEXT, detail TEXT, "
                 "flag_latency_ms REAL)")                               # every raised flag

FLAG_LATENCIES_MS = []                                  # collected for the final stats cell

async def alarm_consumer(alert_queue: asyncio.Queue):
    """STEP 3: await alert -> print INSTANTLY (flush=True) + record to the audit DB."""
    while True:
        alert = await alert_queue.get()                 # sleeps until the Brain raises a flag (or sentinel)
        if alert is None:                               # sentinel = pipeline shutting down
            break
        flag_latency_ms  = (alert["flagged_at"] - alert["transcribed_at"]) * 1000  # end-to-end
        queue_wait_ms    = (alert.get("dequeued_at", alert["transcribed_at"]) - alert["transcribed_at"]) * 1000
        judge_ms         = (alert["flagged_at"] - alert.get("dequeued_at", alert["transcribed_at"])) * 1000
        FLAG_LATENCIES_MS.append(judge_ms)              # the honest real-time number: pure judge latency
        audit_db.execute("INSERT INTO alerts VALUES (?,?,?,?,?)",
                         (alert["call_id"], alert["turn_number"], alert["rule"],
                          alert["detail"], flag_latency_ms))
        print("\n" + "!" * 78, flush=True)              # flush=True -> appears the instant it happens
        print(f"!! ESCALATE  {alert['call_id']}  turn {alert['turn_number']}  ({alert['rule']})", flush=True)
        print(f"!! {alert['detail']}", flush=True)
        print(f"!! judge latency: {judge_ms:.0f} ms  (+{queue_wait_ms:.0f} ms queue wait in "
              f"flat-out benchmark mode; end-to-end {flag_latency_ms:.0f} ms)", flush=True)
        # NOTE: queue wait exists because REALTIME_PACING=False floods all chunks at once.
        # With live pacing (1 chunk / 5s) the queue stays empty whenever judge < 5s.
        print("!" * 78 + "\n", flush=True)
        await emit_event({"type": "alert", "call_id": alert["call_id"],
                          "turn_number": alert["turn_number"], "rule": alert["rule"],
                          "detail": alert["detail"], "judge_ms": round(judge_ms)})   # -> GUI banner

print("THE ALARM ready (instant flush + sqlite :memory: audit)")

THE ALARM ready (instant flush + sqlite :memory: audit)


## CELL 8 — Run the assembly line

Wires everything: one transcript queue + one Ears producer + one Brain worker **per call**,
one shared alert queue + one Alarm consumer, all under a single `asyncio.gather`. With
`REALTIME_PACING=False` this is a flat-out benchmark; set it `True` in CELL 5 to watch flags
fire against a simulated live stream.

In [34]:
# ============ CELL 8: orchestration ============
REALTIME_SINGLE_CALL = True    # ====== THE MODE TOGGLE ======
                               # True  -> ONE call (SELECTED_CALL_ID or random), paced at 1x like a live call
                               # False -> BULK: every recording in the folder, flat-out, full-accuracy ASR
SELECTED_CALL_ID = ""          # single-call mode only; "" -> pick one at random

import random

print("available call ids:", [p.stem for p in audio_files])
REALTIME_PACING = REALTIME_SINGLE_CALL                  # overrides CELL 5's flag — ears_producer
                                                        # reads this global at runtime, so the toggle
                                                        # here is the ONLY switch that matters
if REALTIME_SINGLE_CALL:
    if SELECTED_CALL_ID:
        selected_recordings = [p for p in audio_files if p.stem == SELECTED_CALL_ID]
        assert selected_recordings, f"no recording named '{SELECTED_CALL_ID}' — see the list above"
    else:
        selected_recordings = [random.choice(audio_files)]
    print(f"REAL-TIME single call: {selected_recordings[0].stem} (paced at 1x — runs as long as the call)")
    if WHISPER_MODEL_SIZE == "large-v3":
        print("note: large-v3 ASR can lag the 5s pacing — set WHISPER_MODEL_SIZE='distil-large-v3' "
              "in CELL 5 for tight real-time behavior")
else:
    selected_recordings = list(audio_files)
    print(f"BULK moderation: all {len(selected_recordings)} recordings, flat-out "
          f"(full-file ASR per channel — progress lines print as each channel completes)")
assert selected_recordings, "No audio files found — check CELL 4b output / DATASET_REFERENCE"

alert_queue = asyncio.Queue()                           # shared Brain -> Alarm channel
alarm_task  = asyncio.create_task(alarm_consumer(alert_queue))         # start STEP 3 first (always listening)

pipeline_started = time.perf_counter()                  # total wall-clock timer
per_call_tasks = []
for audio_path in selected_recordings:                  # build one independent lane per call
    call_id = audio_path.stem                           # filename (no extension) as the call's id
    transcript_queue = asyncio.Queue()                  # the zero-copy handshake 1->2 for THIS call
    per_call_tasks.append(ears_producer(call_id, audio_path, transcript_queue))   # STEP 1 (produces + sentinel)
    per_call_tasks.append(brain_worker(call_id, transcript_queue, alert_queue))   # STEP 2 (consumes until sentinel)

await asyncio.gather(*per_call_tasks)                   # run all lanes concurrently; returns when audio is done
await alert_queue.put(None)                             # sentinel: tell the Alarm to finish
await alarm_task                                        # wait for the last alert to be printed/stored
pipeline_seconds = time.perf_counter() - pipeline_started

total_turns = audit_db.execute("SELECT COUNT(*) FROM turns").fetchone()[0]
print(f"assembly line done: {len(selected_recordings)} call(s), {total_turns} judged utterances "
      f"in {pipeline_seconds:.1f}s ({total_turns/max(pipeline_seconds,0.001):.1f} utterances/s)")

available call ids: ['CA0fe99171c6dec5c26dbe5fa5d10c863a', 'CA3e0c1114f78be6bd450860973c404dba', 'CA4950c1c8c305cc85c5f5f040229fe608', 'CA5f229fc25030bdc650d548bdcf95780f', 'CA769e290725c8cb356344c837470375f2']
REAL-TIME single call: CA4950c1c8c305cc85c5f5f040229fe608 (paced at 1x — runs as long as the call)
note: large-v3 ASR can lag the 5s pacing — set WHISPER_MODEL_SIZE='distil-large-v3' in CELL 5 for tight real-time behavior


`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk000] SKIP no confident speech
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk000] SKIP no confident speech
[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk001] SKIP silence
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk001] "The pound key."
[CA4950c1c8c305cc85c5f5f040229fe608 t01]      rep (+0): The pound key.
        timing: asr 370ms | queue 0ms | judge 3301ms | audio->verdict 3671ms
[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk002] SKIP silence
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk002] "We didn't recognize"
[CA4950c1c8c305cc85c5f5f040229fe608 t02]      rep (+0): We didn't recognize
        timing: asr 329ms | queue 0ms | judge 3372ms | audio->verdict 3702ms
[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk003] SKIP silence
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk003] "recognize that entry. We didn't recognize that entry. Please try again."
[CA4950c1c8c305cc85c5f5f040229fe608 t03]      rep (+0): recognize that entry. We didn't recognize

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk009] "Please note that this call is recorded"
[CA4950c1c8c305cc85c5f5f040229fe608 t07]      rep (+0): Please note that this call is recorded
        timing: asr 403ms | queue 0ms | judge 3080ms | audio->verdict 3484ms
[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk010] SKIP no confident speech
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk010] "training and coaching purposes. Please stay on the line while we connect you with the next available information."
[CA4950c1c8c305cc85c5f5f040229fe608 t08]      rep (+0): training and coaching purposes. Please stay on the line while we connect you with the next available
        timing: asr 649ms | queue 0ms | judge 2934ms | audio->verdict 3583ms
[CA4950c1c8c305cc85c5f5f040229fe608 ch0 chunk011] SKIP silence
[CA4950c1c8c305cc85c5f5f040229fe608 ch1 chunk011] "agent."
[CA4950c1c8c305cc85c5f5f040229fe608 t09]      rep (+0): agent.
        timing: asr 240ms | queue 0ms | judge 0ms | audio->verdict 240ms
[CA4

## CELL 9 — Results: flags, latency, tokens

Three readouts, all from in-memory data: the **alerts** (what was flagged, which rule, how
fast), **latency** percentiles per stage (judge latency *is* the flag latency), and exact
**token usage** per stage with a combined total.

In [ ]:
# ============ CELL 9: results ============
print("RAISED FLAGS")                                   # straight from the in-memory audit DB
for row in audit_db.execute("SELECT call_id, turn_number, rule, flag_latency_ms FROM alerts"):
    print(f"  {row[0]:<28} turn {row[1]:<3} {row[2]:<48} {row[3]:>7.0f} ms")
if not FLAG_LATENCIES_MS:
    print("  (no escalations triggered on these recordings)")

def percentile(values, fraction):                       # tiny helper — avoids a numpy dependency here
    ordered = sorted(values)
    return ordered[min(int(len(ordered) * fraction), len(ordered) - 1)]

print("\nLATENCY (ms)")                                 # judge latency == transcript->flag latency per turn
for stage, samples in STAGE_LATENCY_MS.items():
    print(f"  {stage:<22} n={len(samples):<5} avg={sum(samples)/len(samples):>7.0f}  "
          f"p95={percentile(samples, 0.95):>7.0f}  max={max(samples):>7.0f}")
if FLAG_LATENCIES_MS:
    print(f"  {'transcript->flag':<22} n={len(FLAG_LATENCIES_MS):<5} "
          f"avg={sum(FLAG_LATENCIES_MS)/len(FLAG_LATENCIES_MS):>7.0f}  "
          f"max={max(FLAG_LATENCIES_MS):>7.0f}")

print("\nTOKENS BY STAGE")                              # exact counts from vLLM's usage field
grand = {"calls": 0, "prompt": 0, "completion": 0}
for stage, usage in sorted(STAGE_TOKEN_USAGE.items()):
    total = usage["prompt"] + usage["completion"]
    print(f"  {stage:<22} calls={usage['calls']:<5} prompt={usage['prompt']:>8,} "
          f"completion={usage['completion']:>8,} total={total:>9,}")
    for field in grand:
        grand[field] += usage[field]
print(f"  {'COMBINED':<22} calls={grand['calls']:<5} prompt={grand['prompt']:>8,} "
      f"completion={grand['completion']:>8,} total={grand['prompt']+grand['completion']:>9,}")

print("\nPER-CALL SUMMARY")                             # SQL over the in-memory audit trail
for row in audit_db.execute(
    "SELECT call_id, COUNT(*), SUM(violations != ''), MIN(sentiment) FROM turns GROUP BY call_id"):
    print(f"  {row[0]:<28} turns={row[1]:<4} turns_with_violations={row[2] or 0:<3} "
          f"worst_sentiment={row[3]}")

## Design notes

- **Why `asyncio.Queue` and not LangChain/LangGraph:** the pipeline is a fixed 3-stage line —
  the architecture document itself specifies in-process queues. A graph framework would add
  abstraction layers (and milliseconds) between transcript and flag for zero functional gain.
- **Why one queue/worker per call:** FIFO order within a call is required (rule 3 reads
  *consecutive* sentiment), while calls stay parallel and vLLM batches across them. The
  shared `Semaphore(16)` is the global concurrency gate.
- **Where the speed comes from:** silence chunks are dropped before transcription (0 cost),
  the prompt carries only the last 3 utterances, guided JSON kills retries, and the alarm
  consumer prints with `flush=True` the instant the Brain pushes — flag latency is one judge
  call (~hundreds of ms on a 4B model), measured and reported per flag.
- **To go truly live:** replace `ears_producer`'s file loop with a WebSocket/WebRTC receiver
  pushing 5s chunks — every line downstream of `transcript_queue.put(...)` stays identical.
- **Real-time rehearsal:** set `REALTIME_PACING = True` in CELL 5 to pace chunks at 1×.

## CELL 10 — Full transcripts (from the audit DB)

Everything the judge actually saw, per call and in order — the ground truth for debugging
false flags: compare these lines against the audio to spot ASR errors, and against the
alert log to see exactly which words triggered which code.

In [ ]:
# ============ CELL 10: transcript dump ============
for (call_id,) in audit_db.execute("SELECT DISTINCT call_id FROM turns"):
    print("=" * 78); print(f"TRANSCRIPT {call_id}"); print("=" * 78)
    rows = audit_db.execute(
        "SELECT turn_number, role, text, sentiment, violations, reason "
        "FROM turns WHERE call_id = ? ORDER BY turn_number", (call_id,))
    for turn_number, role, text, sentiment, violations, reason in rows:
        flag_note = f"   <- {violations}: {reason}" if violations else ""
        print(f"[t{turn_number:02d}] {role:>8} ({sentiment:+d}): {text}{flag_note}")
    print()

## CELL 11 — Replay the run onto the dashboard

If the GUI was restarted (or down) during a run, the dashboard missed those events.
This cell re-emits **every judged turn and escalation from the audit DB**, repopulating
the dashboard exactly as it would have looked live. Run CELL 3b first if the kernel
still has the old emitter loaded.

In [ ]:
# ============ CELL 11: replay audit DB -> dashboard ============
replayed_turns = 0
for call_id_, turn_no_, role_, text_, sentiment_, violations_, reason_ in audit_db.execute(
        "SELECT call_id, turn_number, role, text, sentiment, violations, reason "
        "FROM turns ORDER BY call_id, turn_number"):
    await emit_event({"type": "turn", "call_id": call_id_, "turn_number": turn_no_,
                      "role": role_, "text": text_, "sentiment": sentiment_,
                      "violations": violations_.split(",") if violations_ else [],
                      "reason": reason_, "asr_ms": 0, "queue_ms": 0, "judge_ms": 0})
    replayed_turns += 1
replayed_alerts = 0
for call_id_, turn_no_, rule_, detail_, latency_ in audit_db.execute(
        "SELECT call_id, turn_number, rule, detail, flag_latency_ms FROM alerts"):
    await emit_event({"type": "alert", "call_id": call_id_, "turn_number": turn_no_,
                      "rule": rule_, "detail": detail_, "judge_ms": round(latency_ or 0)})
    replayed_alerts += 1
print(f"replayed {replayed_turns} turns + {replayed_alerts} alerts to the dashboard "
      f"(GUI_AVAILABLE={GUI_AVAILABLE})")